In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# MLFlow Initialization

In [7]:
%pip install -q dagshub mlflow


Note: you may need to restart the kernel to use updated packages.


In [8]:
import dagshub
dagshub.init(repo_owner='izere23', repo_name='Assignment-2-IEEE-CIS-Fraud-Detection', mlflow=True)

Initialized MLflow to track repo "izere23/Assignment-2-IEEE-CIS-Fraud-Detection"

Repository izere23/Assignment-2-IEEE-CIS-Fraud-Detection initialized!

In [9]:
import mlflow

# Data Loading

In [10]:
BASE_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'
train_transaction = pd.read_csv(BASE_PATH + "train_transaction.csv")
train_identity = pd.read_csv(BASE_PATH + "train_identity.csv")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)
print(train["isFraud"].value_counts(normalize=True))

(590540, 434)
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["isFraud"])
y = train["isFraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

(472432, 433) (118108, 433)
0.03498916246147594 0.0349933958749619


# Cleaning

## High Missing Columns

In [12]:
from sklearn.base import BaseEstimator, TransformerMixin

class HighMissingDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold].tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

## High Dominance Columns

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class HighDominanceDropper(BaseEstimator, TransformerMixin):
    def __init__(self, dominance_threshold):
        self.dominance_threshold = dominance_threshold

    def fit(self, X, y=None):
        X = X.copy()

        dominance = X.apply(
            lambda col: col.value_counts(normalize=True, dropna=False).iloc[0]
            if col.value_counts(dropna=False).shape[0] > 0 else 1
        )

        self.cols_to_drop_ = dominance[
            dominance > self.dominance_threshold
        ].index.tolist()

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

### Cleaning Pipeline

In [14]:
cleaning_pipeline = Pipeline([
    ("high_missing_columns", HighMissingDropper(threshold=0.98))
])

# Preprocessing

In [15]:
class AutoPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, use_scaling=True, min_freq=None):
        self.use_scaling = use_scaling
        self.min_freq = min_freq
        self.preprocessor_ = None

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

        num_steps = [
            ("imputer", SimpleImputer(strategy="median"))
        ]

        if self.use_scaling:
            num_steps.append(("scaler", StandardScaler()))

        num_pipeline = Pipeline(num_steps)

        if self.min_freq is not None:
            ohe = OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=self.min_freq
            )
        else:
            ohe = OneHotEncoder(handle_unknown="ignore")

        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", ohe)
        ])

        self.preprocessor_ = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])

        self.preprocessor_.fit(X, y)
        return self

    def transform(self, X):
        return self.preprocessor_.transform(X)

# Feature Engineering

## Log-Transform

In [16]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class LogAmountAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        return X

In [17]:
feature_engineering_pipeline = Pipeline([
    ("log_amount", LogAmountAdder())
])

# Feature Selection

## L1 selector

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression


class L1FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, C=0.01):
        self.C = C
        self.selector_ = None

    def fit(self, X, y=None):
        l1_model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=self.C,
            class_weight="balanced",
            random_state=42,
            max_iter=500
        )

        self.selector_ = SelectFromModel(l1_model)
        self.selector_.fit(X, y)

        return self

    def transform(self, X):
        return self.selector_.transform(X)

## Corelation Filter

In [19]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import sparse
import pandas as pd
import numpy as np


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):

        if sparse.issparse(X):
            X = X.toarray()

        X_df = pd.DataFrame(X)

        corr_matrix = X_df.corr().abs()

        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        self.to_drop_ = [
            column for column in upper.columns
            if any(upper[column] > self.threshold)
        ]

        self.to_keep_ = [
            col for col in X_df.columns
            if col not in self.to_drop_
        ]

        return self

    def transform(self, X):

        if sparse.issparse(X):
            X = X.toarray()

        return X[:, self.to_keep_]

## RFE

In [20]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression


class LRRFESelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_features_to_select=250, step=0.3):
        self.n_features_to_select = n_features_to_select
        self.step = step
        self.selector_ = None

    def fit(self, X, y=None):
        estimator = LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            solver="liblinear",
            penalty="l2",
            C=0.1,
            random_state=42
        )

        self.selector_ = RFE(
            estimator=estimator,
            n_features_to_select=self.n_features_to_select,
            step=self.step
        )

        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        return self.selector_.transform(X)

### Feature Selection pipeline

In [21]:
feature_selection_pipeline = Pipeline([
    ("rfe_selector", LRRFESelector(
        n_features_to_select=250,
        step=0.3
    ))
])

# Linear Regression Pipeline

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    solver="liblinear",
    penalty="l2",
    C=0.1,
    random_state=42
)

pipeline = Pipeline([
    ("feature_engineering", feature_engineering_pipeline),
    ("cleaning", cleaning_pipeline),
    ("preprocessing", AutoPreprocessor(use_scaling=True, min_freq=None)),
    ("feature_selection", feature_selection_pipeline),
    ("model", model)
])

# Training

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, log_loss

mlflow.set_experiment("LogisticRegression_Training")

with mlflow.start_run(run_name="LR_log_RFE_250_step03"):

    threshold = 0.5
    
    pipeline.fit(X_train, y_train)

    train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
    val_pred_proba = pipeline.predict_proba(X_val)[:, 1]

    val_pred = (val_pred_proba >= threshold).astype(int)

    train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    val_roc_auc = roc_auc_score(y_val, val_pred_proba)

    train_log_loss = log_loss(y_train, train_pred_proba)
    val_log_loss = log_loss(y_val, val_pred_proba)

    overfit_gap = train_roc_auc - val_roc_auc
    overfit_loss_gap = val_log_loss - train_log_loss

    precision = precision_score(y_val, val_pred)
    recall = recall_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("baseline_type", "baseline")
    mlflow.log_param("numeric_imputation", "median")
    mlflow.log_param("categorical_imputation", "most_frequent")
    mlflow.log_param("encoding", "OHE")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 500)
    mlflow.log_param("threshold", threshold)
    mlflow.log_param("solver", "liblinear")
    mlflow.log_param("regularization", "L2")
    mlflow.log_param("C", 0.1)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("missing_threshold", 0.98)
    mlflow.log_param("dominacne_threshold", 0.995)
    mlflow.log_param("feature_engineering", "True")
    mlflow.log_param("engineered_feature", "Log-Transform")
    mlflow.log_param("feature_selection", "True")
    mlflow.log_param("selected_feature", "RFE")
    
    mlflow.log_metric("train_roc_auc", train_roc_auc)
    mlflow.log_metric("val_roc_auc", val_roc_auc)
    mlflow.log_metric("overfit_gap", overfit_gap)

    mlflow.log_metric("train_log_loss", train_log_loss)
    mlflow.log_metric("val_log_loss", val_log_loss)
    mlflow.log_metric("overfit_loss_gap", overfit_loss_gap)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="logistic_regression"
    )

    print("Train ROC-AUC:", train_roc_auc)
    print("Val ROC-AUC:", val_roc_auc)
    print("Overfit Gap:", overfit_gap)
    print("Train Log Loss:", train_log_loss)
    print("Val Log Loss:", val_log_loss)
    print("Overfit Loss Gap:", overfit_loss_gap)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

In [ ]:
# #Best So Far
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline

# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline

# model = LogisticRegression(
#     max_iter=500,
#     class_weight="balanced",
#     solver="liblinear",
#     penalty="l2",
#     C=0.1,
#     random_state=42
# )

# pipeline = Pipeline([
#     ("feature_engineering", feature_engineering_pipeline),
#     ("drop_high_missing", HighMissingDropper(threshold=0.98)),
#     ("preprocessing", AutoPreprocessor(use_scaling=True, min_freq=None)),
#     ("model", model)
# ])